# Aesthetic Image Scoring — Experiments

All logic lives in `src/`. This notebook only runs experiments and collects
results, so re-running it top to bottom is cheap and always reproduces the
table at the bottom.

Run it from the project root with the project environment selected as the
kernel. No GPU is required.

## Setup

Fetches the AADB label file if it is not already present, and reports the
device that feature extraction will use.

The image archive is the one piece that cannot be automated: the authors'
Google Drive links predate the current sharing format and no downloader can
resolve them. Fetch `datasetImages_warp256.zip` by hand and unzip it to
`data/`.

In [ ]:
import os

LABELS = 'data/AADBinfo.mat'
LABEL_URL = ('https://raw.githubusercontent.com/aimerykong/'
             'deepImageAestheticsAnalysis/master/AADBinfo.mat')

# 175 KB, the only file needed from the authors' repository.
if not os.path.exists(LABELS):
    !mkdir -p data && curl -sLo {LABELS} {LABEL_URL}

from src.backbones import pick_device
print("device:", pick_device())

## Feature extraction

Run once per backbone. Each call writes `{backbone}_train.npy` and
`{backbone}_test.npy` into `features/` and skips the work entirely if those
files already exist.

The device is picked automatically: CUDA, then Apple MPS, then CPU. Expect
roughly two minutes per backbone on Apple silicon and closer to twenty on
CPU.

In [ ]:
IMAGES = 'data/datasetImages_warp256'   # adjust to your unzipped archive

!python -m src.extract_features --backbone clip     --images {IMAGES}
!python -m src.extract_features --backbone resnet50 --images {IMAGES}

## Experiments

`run_experiment` loads the cached features, holds out 10% of training data for
validation, fits the head with early stopping on validation SRCC, and evaluates
on the test set once.

Each configuration below isolates one variable, so the differences between rows
are interpretable rather than confounded.

In [ ]:
from src.train import run_experiment

results = {}

results['CLIP ViT-B/32'] = run_experiment(['clip'], 'features')

### CNN branch

ResNet-50 features alone, for comparison. Expect this to trail CLIP: ImageNet
pretraining optimises for object identity, which overlaps with aesthetic
judgement but is not the same thing.

In [ ]:
results['ResNet-50'] = run_experiment(['resnet50'], 'features')

### Late fusion

Both views together. Whether fusion helps is an empirical question — if the two
backbones encode largely the same information about aesthetics, the extra
parameters buy nothing and may cost a little through overfitting.

In [ ]:
results['CLIP + ResNet-50'] = run_experiment(['clip', 'resnet50'], 'features')

### Ranking loss

The metrics measure ordering while MSE measures absolute error. Adding a
pairwise margin term optimises the ranking objective directly.

In [ ]:
results['CLIP + rank loss'] = run_experiment(['clip'], 'features', loss_name='mse+rank')
results['Fusion + rank loss'] = run_experiment(
    ['clip', 'resnet50'], 'features', loss_name='mse+rank'
)

## Ablation table

In [ ]:
import pandas as pd

df = (pd.DataFrame(results).T[['SRCC', 'KRCC', 'MSE', 'best_epoch']]
        .sort_values('SRCC', ascending=False)
        .round(4))
df

Reference point: the original AADB paper (Kong et al., ECCV 2016) reports SRCC
0.678 on this test set.

Read any improvement over it as the contribution of large-scale pretraining
rather than of a better head — CLIP's 400M image-text pairs were not available
in 2016, and the head here is deliberately minimal.

Validation and test scores should sit close together. A large gap in either
direction is a symptom rather than a result, and usually means a stale model was
reused across runs or the caches fell out of sync.

## Robustness check

The differences above are one to two points of SRCC, and a single seed can
flatter or punish a configuration by that much on its own. This cell re-runs
every configuration across ten seeds and tests the gaps properly.

A seed fixes the train/validation split, so the same seed means the same split
for every configuration. That makes the comparison *paired*, which removes the
split-to-split variance that dominates the raw standard deviations — the right
test is on the per-seed differences, not on the two means.

Takes roughly three minutes; it is fifty training runs on cached features.

In [ ]:
import io, contextlib
import numpy as np
from scipy.stats import ttest_rel

CONFIGS = [('CLIP',          ['clip'],             'mse'),
           ('CLIP + rank',   ['clip'],             'mse+rank'),
           ('Fusion',        ['clip', 'resnet50'], 'mse'),
           ('Fusion + rank', ['clip', 'resnet50'], 'mse+rank'),
           ('ResNet-50',     ['resnet50'],         'mse')]
SEEDS = range(10)

srcc = {}
for name, feats, loss in CONFIGS:
    runs = []
    for s in SEEDS:
        with contextlib.redirect_stdout(io.StringIO()):   # one line per config, not per run
            runs.append(run_experiment(feats, 'features', seed=s, loss_name=loss)['SRCC'])
    srcc[name] = np.array(runs)
    print(f"{name:15s} {np.mean(runs):.4f} +/- {np.std(runs, ddof=1):.4f}")

base = srcc['CLIP']
print(f"\nPaired against CLIP over {len(list(SEEDS))} seeds")
for name, runs in srcc.items():
    if name == 'CLIP':
        continue
    delta = base - runs
    print(f"{name:15s} {delta.mean():+.4f}  p={ttest_rel(base, runs).pvalue:.4f}  "
          f"CLIP ahead on {int((delta > 0).sum())}/{len(delta)} seeds")